# {Project Title}📝
 Which Vulnerabilities Do Attackers Actually Exploit?
![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
📝 Security teams can't patch every vulnerability, so they need to know which ones attackers actually exploit.

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
1. Which CVE characteristics (CVSS severity, vendor, weakness type, age) are associated with being listed as actively exploited?
2. Does EPSS tell us more about exploitation than CVSS does?
3. How long does it take from a CVE's publication to its KEV listing, and does it vary by vendor or weakness type?

## What would an answer look like?
*What is your hypothesized answer to your question?*
1. Bar chart: share of CVEs in KEV by CVSS band
2. Scatter plot: EPSS vs. CVSS, with KEV CVEs highlighted
3. Box plot: days from publication to KEV listing, by vendor
4. Table: top weakness types (CWE) among KEV CVEs

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
Data Sources
NVD CVE API
CISA KEV catalog
FIRST EPSS scores
All three share the CVE ID. NVD is the base table, left-joined with KEV and EPSS. A CVE missing from KEV is treated as "not known exploited."


## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*

- Loaded each source with its own function (`load_kev`, `load_epss`, `load_nvd`).
- Pulled NVD CVEs published in August 2026 through the paged API (12,719 CVEs) and flattened each record into one row: ID, publish date, status, CVSS score and source, CWE, and vendor.
- Left-joined NVD with EPSS and KEV on CVE ID. The row count stayed at 12,719; 12,258 matched EPSS (the 461 that didn't are all `Rejected` CVEs) and 20 matched KEV.

**Limitations**
- August alone gives only 20 KEV matches (about 0.16%), so the next step is widening the date range to several years.
- About 11% of CVEs have no CVSS score, because NVD hasn't analyzed them yet. Most scores come from the CVE publisher, not NVD.
- EPSS is a current snapshot, and KEV's `dateAdded` is when CISA listed the CVE, not when exploitation began.

In [1]:
import time
import pandas as pd
import requests

KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
EPSS_URL = "https://epss.empiricalsecurity.com/epss_scores-current.csv.gz"
NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

In [2]:
   def load_kev(url):
       """Download the CISA KEV catalog and return it as a DataFrame."""
       response = requests.get(url, timeout=30)
       response.raise_for_status()
       data = response.json()
       return pd.json_normalize(data["vulnerabilities"])  

   kev = load_kev(KEV_URL)
   kev.head()  


,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,forensicTriage,notes,cwes
0,CVE-2026-76504,Cisco,Catalyst SD-WAN Manager,Cisco Catalyst SD-WAN Manager Hex Encoding Vul...,2026-09-30,Cisco Catalyst SD-WAN Manager contains a hex e...,Apply mitigations in accordance with vendor in...,2026-10-03,Unknown,Yes,https://sec.cloudapps.cisco.com/security/cente...,[CWE-177]
1,CVE-2026-86950,Apple,Multiple Products,Apple Multiple Products Out-of-Bounds Write Vu...,2026-09-29,"Apple iOS, macOS, and iPadOS contain an out-of...",Apply mitigations in accordance with vendor in...,2026-10-02,Unknown,Yes,https://support.apple.com/en-us/149226 ; https...,[CWE-787]
2,CVE-2026-88772,Citrix,NetScaler,Citrix NetScaler Improper Restriction of Opera...,2026-09-27,Citrix NetScaler ADC and NetScaler Gateway con...,Apply mitigations in accordance with vendor in...,2026-09-30,Unknown,Yes,Running the provided IOCs in the NetScaler con...,[CWE-119]
3,CVE-2026-88771,Citrix,NetScaler,Citrix NetScaler Improper Input Validation Vul...,2026-09-27,Citrix NetScaler ADC and NetScaler Gateway con...,Apply mitigations in accordance with vendor in...,2026-09-30,Unknown,Yes,Running the provided IOCs in the NetScaler con...,[CWE-119]
4,CVE-2026-67279,MikroTik,RouterOS,Mikrotik RouterOS Improper Enforcement of Beha...,2026-09-25,Mikrotik RouterOS contains an improper enforce...,Apply mitigations in accordance with vendor in...,2026-09-28,Unknown,No,https://mikrotik.com/supportsec/september-2026...,[CWE-841]


In [3]:
print(kev.shape)
print(kev.columns.tolist())


(1730, 12)
['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded', 'shortDescription', 'requiredAction', 'dueDate', 'knownRansomwareCampaignUse', 'forensicTriage', 'notes', 'cwes']


In [4]:
def load_epss(url):
    """Download the current EPSS scores and return them as a DataFrame."""
    return pd.read_csv(url, skiprows=1)  # the file starts with a comment line

epss = load_epss(EPSS_URL)
print(epss.shape)
epss.head()


(381189, 3)


,cve,epss,percentile
0,CVE-1999-0001,0.03351,0.88271
1,CVE-1999-0002,0.27858,0.98039
2,CVE-1999-0003,0.24565,0.97815
3,CVE-1999-0004,0.02794,0.85912
4,CVE-1999-0005,0.18415,0.97148


In [5]:
def get_cvss(cve):
    """Return (baseScore, source type) from the CVSS v3.1 metrics, or (None, None)."""
    entries = cve.get("metrics", {}).get("cvssMetricV31", [])
    if not entries:
        return None, None
    # prefer the Primary (NVD) entry if there is one
    for entry in entries:
        if entry["type"] == "Primary":
            return entry["cvssData"]["baseScore"], entry["type"]
    return entries[0]["cvssData"]["baseScore"], entries[0]["type"]



In [6]:
def get_cwe(cve):
    """Return the first CWE ID (e.g. 'CWE-79'), or None if the CVE has no weakness data."""
    weaknesses = cve.get("weaknesses", [])
    if not weaknesses:
        return None
    return weaknesses[0]["description"][0]["value"]  



In [7]:


def get_vendor(cve):
    """Return the first vendor listed by the CVE's publisher, or None."""
    affected = cve.get("affected", [])
    if not affected or not affected[0].get("affectedData"):
        return None
    return affected[0]["affectedData"][0].get("vendor")

def parse_cve(cve):
    """Flatten one NVD CVE record into a single row."""
    score, score_type = get_cvss(cve)
    return {
        "cve_id": cve["id"],
        "published": cve["published"],
        "vuln_status": cve["vulnStatus"],
        "cvss_score": score,
        "cvss_source": score_type,
        "cwe": get_cwe(cve),
        "vendor": get_vendor(cve),
    }

def load_nvd(start, end, max_pages=None):
    """Page through the NVD API for CVEs published between start and end."""
    rows = []
    start_index = 0
    page = 0
    while True:
        params = {
            "pubStartDate": start,
            "pubEndDate": end,
            "resultsPerPage": 2000,
            "startIndex": start_index,
        }
        response = requests.get(NVD_URL, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        rows.extend(parse_cve(item["cve"]) for item in data["vulnerabilities"])

        start_index += data["resultsPerPage"]  
        page += 1
        if start_index >= data["totalResults"] or (max_pages and page >= max_pages):
            break
        time.sleep(6)  # stay under the no-API-key rate limit
    return pd.DataFrame(rows)


In [8]:
nvd = load_nvd("2026-08-01T00:00:00.000", "2026-08-31T23:59:59.999", max_pages=1)
print(nvd.shape)
nvd.head()

(2000, 7)


,cve_id,published,vuln_status,cvss_score,cvss_source,cwe,vendor
0,CVE-2026-13362,2026-08-01T03:16:25.307,Deferred,6.4,Secondary,CWE-79,sendpulse
1,CVE-2026-15006,2026-08-01T03:16:25.460,Deferred,7.5,Secondary,CWE-22,bitpressadmin
2,CVE-2026-15403,2026-08-01T03:16:25.620,Deferred,4.9,Secondary,CWE-89,dotonpaper
3,CVE-2026-15414,2026-08-01T03:16:25.757,Deferred,8.8,Secondary,CWE-269,wpswings
4,CVE-2026-7623,2026-08-01T03:16:26.070,Deferred,6.4,Secondary,CWE-79,brainstormforce


In [9]:
print(nvd.shape)
print(nvd["vuln_status"].value_counts())
print(nvd["cvss_score"].isna().mean())

(2000, 7)
vuln_status
Deferred               1107
Analyzed                495
Awaiting Analysis       241
Modified                 46
Received                 46
Rejected                 41
Undergoing Analysis      24
Name: count, dtype: int64
0.1145


In [10]:
print(nvd[nvd["cvss_score"].isna()]["vuln_status"].value_counts())

vuln_status
Deferred               90
Awaiting Analysis      78
Rejected               41
Received               19
Undergoing Analysis     1
Name: count, dtype: int64


In [11]:
   nvd = load_nvd("2026-08-01T00:00:00.000", "2026-08-31T23:59:59.999")

In [12]:
   print(nvd.shape)
   print(nvd["cve_id"].is_unique)

(12719, 7)
True


In [13]:
merged = (
    nvd
    .merge(epss, left_on="cve_id", right_on="cve", how="left")
    .merge(kev[["cveID", "dateAdded"]], left_on="cve_id", right_on="cveID", how="left")
)
print(merged.shape)
print("EPSS matched:", merged["epss"].notna().sum())
print("KEV matched:", merged["dateAdded"].notna().sum())

(12719, 12)
EPSS matched: 12258
KEV matched: 20


In [14]:
unmatched = merged[merged["epss"].isna()]
print(unmatched["vuln_status"].value_counts())

vuln_status
Rejected    461
Name: count, dtype: int64


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

### Data sources
- NIST National Vulnerability Database (NVD) CVE API 2.0: https://nvd.nist.gov/developers/vulnerabilities (accessed Sep. 30, 2026)
- CISA Known Exploited Vulnerabilities (KEV) Catalog: https://www.cisa.gov/known-exploited-vulnerabilities-catalog (accessed Sep. 30, 2026)
- FIRST Exploit Prediction Scoring System (EPSS): https://www.first.org/epss/ (accessed Sep. 30, 2026)

### Documentation
- pandas documentation (`read_csv`, `json_normalize`, `merge`): https://pandas.pydata.org/docs/
- Requests library documentation: https://requests.readthedocs.io/